<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Autoencoders and reconstruction error

**[Autoencoders and reconstruction error](https://learning.nextia-ai.com/courses/anomaly-detection/m06/autoencoders-and-reconstruction-error/)** · Anomaly Detection: Find What Is Unusual · Module 6, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train a tiny autoencoder on a CPU, use its reconstruction error as an anomaly score, and compare it honestly with Isolation Forest in the comparison table.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB, and PyTorch: about 200 MB). No account, no GPU. Training takes a few seconds on a CPU. |
| **You should know** | Isolation Forest and the behaviour features, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/) and [Features decide more than the algorithm](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/). Neural networks, from the deep learning course (helpful, not required). |
| **You do not need** | A local project, or a GPU. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m06/autoencoders-and-reconstruction-error/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M06-L01-autoencoders-and-reconstruction-error/autoencoder-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, torch 2.15.0. It needs an internet connection. The first time, it takes up to a minute. PyTorch is large (about 200 MB): the first time, this cell can take two or three minutes.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 torch==2.15.0
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'torch']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Rebuild what is normal

Think of a person who copies signatures at a bank all day. After a while, they can redraw a customer's usual signature from memory. Give them a strange signature, and their copy will look different from it: they only know how to draw the usual one.

An **autoencoder** is a small neural network that does this with numbers. It squeezes each input into fewer numbers (the **code**) and rebuilds the input from them. It learns only from normal data, so it rebuilds normal data well. The difference between the input and the rebuilt input is the **reconstruction error**: small for normal points, large for points unlike anything it learned.

Run the next cell. Five normal points lie near the line y = 2x; the sixth, (2, 0.5), does not. The autoencoder squeezes 2 numbers into 1 and learns from the five normal points only.

> **Check.** You should see errors below 0.001 for the first five points and about 1.59 for the last one.

In [ ]:
import torch
torch.manual_seed(0)
torch.set_num_threads(1)                 # one thread: the same numbers on every run
# normal points lie close to the line y = 2x; the last point does not
X = torch.tensor([[0.0, 0.0], [1.0, 2.0], [2.0, 4.1], [3.0, 5.9], [4.0, 8.0], [2.0, 0.5]])
encoder = torch.nn.Linear(2, 1)          # squeeze 2 numbers into 1
decoder = torch.nn.Linear(1, 2)          # and rebuild 2 numbers from it
net = torch.nn.Sequential(encoder, decoder)
opt = torch.optim.Adam(net.parameters(), lr=0.05)
normal = X[:5]                           # it learns from the normal points only
for step in range(2000):
    loss = ((net(normal) - normal) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    tiny_error = ((net(X) - X) ** 2).mean(dim=1)
print(pd.DataFrame({"x": X[:, 0], "y": X[:, 1], "rebuilt x": net(X)[:, 0].detach().round(decimals=2),
                    "rebuilt y": net(X)[:, 1].detach().round(decimals=2), "error": tiny_error.round(decimals=3)}).to_string(index=False))

One number is enough to describe a point on the line (how far along it is), so the network rebuilds the normal points almost exactly. The sixth point is off the line: the network rebuilds it **onto** the line, and the difference is the score.

## 2. The payments

The next cell scales the same behaviour features as Isolation Forest in Module 4, and makes PyTorch tensors of the reference and the evaluation months.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
X_ref = torch.tensor(scaler.transform(behaviour.loc[reference.index]), dtype=torch.float32)
X_eval = torch.tensor(scaler.transform(behaviour.loc[evaluation.index]), dtype=torch.float32)
print(X_ref.shape, X_eval.shape)

Run the next cell. It trains an autoencoder of 8 → 16 → 3 → 16 → 8 numbers (about 400 weights) for 10 passes over the reference payments, and computes each evaluation payment's error. On a laptop CPU it takes a few seconds.

In [ ]:
def train_autoencoder(X, hidden=16, code_size=3, epochs=10, seed=0):
    """A small autoencoder: 8 numbers -> 16 -> 3 -> 16 -> 8. It learns from the reference payments only."""
    torch.manual_seed(seed)
    d = X.shape[1]
    net = torch.nn.Sequential(torch.nn.Linear(d, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, code_size),
                              torch.nn.ReLU(), torch.nn.Linear(code_size, hidden), torch.nn.ReLU(),
                              torch.nn.Linear(hidden, d))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        perm = torch.randperm(len(X), generator=gen)
        for i in range(0, len(X), 512):
            batch = X[perm[i:i + 512]]
            loss = ((net(batch) - batch) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch + 1}: loss {loss.item():.4f}")
    return net

net = train_autoencoder(X_ref)
with torch.no_grad():
    error = ((net(X_eval) - X_eval) ** 2).mean(dim=1).numpy()

## 3. Predict, then run: the network against the forest

> **Predict.** Tomás expects the neural network to find more. Will the autoencoder's precision at the budget be higher or lower than Isolation Forest's, on the same features and the same data?

In [ ]:
table = compare.Table()
table.add(compare.score_payments(evaluation, error, "autoencoder", "behaviour features; 8-16-3-16-8, 10 epochs",
                                 explain_minutes=5))
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(X_ref.numpy())
table.add(compare.score_payments(evaluation, -forest.score_samples(X_eval.numpy()), "Isolation Forest",
                                 "behaviour features; 200 trees", explain_minutes=3))
table.show()

> **Check.** You should see 0.434 for the autoencoder and 0.500 for Isolation Forest.

On the same data, the autoencoder is lower. It also took more code, more choices (layers, epochs, learning rate) and more time to explain an alert. Run the next cell to see which kinds of fraud each one finds.

In [ ]:
print("autoencoder:", compare.recall_by_kind(evaluation, error).to_dict())
print("Isolation Forest:", compare.recall_by_kind(evaluation, -forest.score_samples(X_eval.numpy())).to_dict())

Each detector is better on some kinds. A table row hides this: look at the kinds before you choose.

## 4. A common mistake: no squeeze

If the code is as wide as the input (8 numbers in, 8 numbers in the middle), the network can learn to copy every input, normal or not. Then every error is small, and the score says little.

## 5. Check yourself

**Your turn 1.** In the tiny example, which point has the largest reconstruction error? Write its position (0 to 5) as `largest`.

In [ ]:
largest = ...   # your answer

In [ ]:
expect('largest', largest, 5)

**Your turn 2.** Train the same autoencoder with `code_size=8` (no squeeze) and score it as `wide_row` (setting `"8-16-8-16-8"`). Compare its precision with the squeezed one.

In [ ]:
wide_row = ...   # your code

In [ ]:
expect_hash('the precision of the wide autoencoder', wide_row['precision at budget'] if isinstance(wide_row, dict) else None, 'a114ff245f4ee3c5')

In the solution: 0.434 without the squeeze, against 0.434 with it.

## Recap

- An autoencoder learns to rebuild normal data; its reconstruction error is an anomaly score. It needs no labels.
- On Larkfield's payments, with the same features, it did not beat Isolation Forest at the budget (0.434 against 0.500), and it costs more to build and to explain.
- A neural network is not a better detector by default: the comparison table decides.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Unusual text and logs with embeddings](https://learning.nextia-ai.com/courses/anomaly-detection/m06/unusual-text-and-logs-with-embeddings/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m06/autoencoders-and-reconstruction-error/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.